> **Paths:** set `ILD_MEDGIFT_ROOT`, `ILD_LUNG_MASK_BASE`, `ILD_MODELS_DIR`, and optionally `ILD_EXPORTS_DIR` in the environment (or a local untracked env file). No machine-local defaults are shipped.

# Phase 2: Multi-Head Fine-Tuning
Conservative binary-first strategy. Loads best Phase 1 checkpoint,
trains hier + path heads while preserving binary.

**Protocol:**
- Phase 2a (ep 0-4): hier + path heads only, binary frozen, lr=1e-4
- Phase 2b (ep 5-14): unfreeze layer4, binary frozen, enc_lr=3e-5 head_lr=1e-4
- Phase 2c (ep 15-19): all heads, lr=1e-5, safety revert if binary < 0.85
- Cascade on 113 patients after training

**Expected time:** ~30-50 min on RTX 3060

In [ ]:
import os, sys, json, gc, re, math, random, warnings
from pathlib import Path
from copy import deepcopy
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
import pydicom
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, accuracy_score

warnings.filterwarnings('ignore')

# ── Detect env ──
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
GLOBAL_SEED = 42
random.seed(GLOBAL_SEED)
np.random.seed(GLOBAL_SEED)
torch.manual_seed(GLOBAL_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(GLOBAL_SEED)
print(f'Device: {device}')

# ── Paths ──
MEDGIFT_ROOT = os.environ.get('ILD_MEDGIFT_ROOT', '')
MODELS_DIR = Path(os.environ.get('ILD_MODELS_DIR', ''))
EXPORTS_DIR = Path(os.environ.get('ILD_EXPORTS_DIR', r'Results/exports_3d_seg'))
if not EXPORTS_DIR.is_absolute():
    EXPORTS_DIR = Path.cwd() / EXPORTS_DIR
CASCADE_DIR = EXPORTS_DIR / 'cascade_maps'
LUNG_MASK_BASE = os.environ.get('ILD_LUNG_MASK_BASE', '').strip()
if not os.path.isdir(LUNG_MASK_BASE):
    LUNG_MASK_BASE = None

for d in (MODELS_DIR, EXPORTS_DIR, CASCADE_DIR):
    os.makedirs(d, exist_ok=True)

# ── Config ──
CLS_PATCH_SIZE = (16, 64, 64)
INFER_DENSE_STRIDE = (4, 8, 8)
FEAT_BATCH_SIZE = 8
GRAD_ACCUM_STEPS = 1
PATCHES_PER_EPOCH = 2400
VAL_PATCHES = 300
PATCH_AUGMENT = True
FINETUNE_WD = 1e-4
FINETUNE_PATIENCE = 8
HU_CLIP = (-1350.0, 150.0)
N_FOLDS = 5
NUM_WORKERS = 0
HEAD_DROPOUT = 0.4
INPLANE = 128
VOL_CACHE_PATIENTS = 6
CASCADE_MAX_PATIENTS = 0  # all
CASCADE_SAVE_MAPS = True
INFER_MAX_PATCHES = 8000
INFER_CLEANUP_EVERY = 64

# Match NB01 dominant patch mining gates
PATCH_LABEL_MODE = 'dominant'
MIN_PATHOLOGY_VOXELS = 80
MIN_TARGET_CLASS_FRAC = 0.15
MIN_PATCH_LUNG_FRAC = 0.20
MIN_NORMAL_LUNG_FRAC = 0.50
MIN_PATCHES_PER_CLASS = 60
PATHOLOGY_PATCH_QUOTA = 0.85
MAX_CLASS_ORIGIN_POOL = 1024

print(f'MEDGIFT_ROOT: {MEDGIFT_ROOT} | exists={os.path.isdir(MEDGIFT_ROOT)}')
print(f'MODELS_DIR: {MODELS_DIR} | exists={os.path.isdir(MODELS_DIR)}')
print(f'LUNG_MASK_BASE: {LUNG_MASK_BASE}')

# ── Class mappings ──
ORIGINAL_CLASS_NAMES = ['Normal', 'Emphysema', 'Fibrosis', 'Ground Glass', 'Micronodules', 'Consolidation']
SEG_NUM_CLASSES = 6
N_BINARY_CLASSES = 2
BINARY_CLASSES = ['Normal', 'ILD']
N_HIER_CLASSES = 3
HIERARCHY_CLASSES = ['Normal', 'Fibrotic', 'NonFibrotic']
HIERARCHY_MAP = {0: 0, 1: 2, 2: 1, 3: 2, 4: 2, 5: 1}
N_PATH_CLASSES = 5
PATHOLOGY_CLASSES = ORIGINAL_CLASS_NAMES[1:]
MEDGIFT_TO_CLS_FALLBACK = {1:0, 2:1, 3:3, 4:2, 5:4, 6:5, 8:5, 11:4, 14:2}
CASCADE_PATH_THRESH = 0.005
CASCADE_PROB_THRESH = 0.45
print(f'Config loaded. PATCH_LABEL_MODE={PATCH_LABEL_MODE} (same as NB01)')

In [ ]:
# ── Model: HierarchicalEncoder3D ──
def _gn(ch, num_groups=8):
    for g in (num_groups, 4, 2, 1):
        if ch % g == 0: return nn.GroupNorm(g, ch)
    return nn.GroupNorm(1, ch)

class SEBlock3D(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        self.fc = nn.Sequential(
            nn.AdaptiveAvgPool3d(1),
            nn.Conv3d(channels, channels // reduction, kernel_size=1),
            nn.ReLU(inplace=True),
            nn.Conv3d(channels // reduction, channels, kernel_size=1),
            nn.Sigmoid())
    def forward(self, x):
        return x * self.fc(x)

class SE_ResBlock3D(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1):
        super().__init__()
        self.conv1 = nn.Conv3d(in_ch, out_ch, 3, stride=stride, padding=1, bias=False)
        self.bn1 = _gn(out_ch)
        self.conv2 = nn.Conv3d(out_ch, out_ch, 3, padding=1, bias=False)
        self.bn2 = _gn(out_ch)
        self.se = SEBlock3D(out_ch)
        self.skip = nn.Identity()
        if in_ch != out_ch or stride != 1:
            self.skip = nn.Sequential(nn.Conv3d(in_ch, out_ch, 1, stride=stride, bias=False), _gn(out_ch))
    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)), inplace=True)
        out = self.bn2(self.conv2(out))
        out = self.se(out)
        return F.relu(out + self.skip(x), inplace=True)

class HierarchicalEncoder3D(nn.Module):
    def __init__(self, in_ch=1, use_se=True):
        super().__init__()
        b = SE_ResBlock3D if use_se else ResBlock3D
        self.stem = nn.Sequential(
            nn.Conv3d(in_ch, 64, kernel_size=(1,7,7), stride=(1,2,2), padding=(0,3,3), bias=False),
            nn.GroupNorm(8, 64), nn.ReLU(inplace=True),
            nn.MaxPool3d(kernel_size=(1,3,3), stride=(1,2,2), padding=(0,1,1)))
        self.layer1 = self._make_layer(64, 64, 2, b)
        self.layer2 = self._make_layer(64, 128, 2, b, stride=2)
        self.layer3 = self._make_layer(128, 256, 2, b, stride=2)
        self.layer4 = self._make_layer(256, 512, 2, b, stride=2)
        self.avgpool = nn.AdaptiveAvgPool3d((1,1,1))
        self.feat_dim = 512
        self.binary_head = nn.Sequential(nn.Dropout(HEAD_DROPOUT), nn.Linear(512, N_BINARY_CLASSES))
        self.hier_head = nn.Sequential(nn.Dropout(HEAD_DROPOUT), nn.Linear(512, N_HIER_CLASSES))
        self.path_head = nn.Sequential(nn.Dropout(HEAD_DROPOUT), nn.Linear(512, N_PATH_CLASSES))
    @staticmethod
    def _make_layer(in_ch, out_ch, blocks, block, stride=1):
        layers = [block(in_ch, out_ch, stride=stride)]
        for _ in range(1, blocks): layers.append(block(out_ch, out_ch))
        return nn.Sequential(*layers)
    def extract_features(self, x):
        x = self.stem(x); x = self.layer1(x); x = self.layer2(x)
        x = self.layer3(x); x = self.layer4(x)
        return self.avgpool(x).flatten(1)
    def forward(self, x, head='binary'):
        f = self.extract_features(x)
        if head == 'binary': return self.binary_head(f)
        elif head == 'hier': return self.hier_head(f)
        elif head == 'path': return self.path_head(f)
        return f

def set_trainable_blocks(model, unfreeze):
    for p in model.parameters(): p.requires_grad = False
    for name in unfreeze:
        m = getattr(model, name, None)
        if m is not None:
            for p in m.parameters(): p.requires_grad = True

# --- Loss functions ---
class WeightedFocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
    def forward(self, logits, targets):
        ce = F.cross_entropy(logits, targets, reduction='none', weight=self.alpha)
        pt = torch.exp(-ce)
        return (ce * (1 - pt) ** self.gamma).mean()

class LabelSmoothCrossEntropy(nn.Module):
    def __init__(self, smoothing=0.1):
        super().__init__()
        self.smoothing = smoothing
    def forward(self, logits, targets):
        n = logits.size(1)
        log_probs = F.log_softmax(logits, dim=1)
        with torch.no_grad():
            smooth = torch.full_like(log_probs, self.smoothing / (n - 1))
            smooth.scatter_(1, targets.unsqueeze(1), 1 - self.smoothing)
        return -(smooth * log_probs).sum(dim=1).mean()

print('Model classes defined.')

In [ ]:
# ── DICOM/Data I/O ──
def dicom_to_hu(ds):
    arr = ds.pixel_array.astype(np.float32)
    slope = float(getattr(ds, 'RescaleSlope', 1.0) or 1.0)
    intercept = float(getattr(ds, 'RescaleIntercept', 0.0) or 0.0)
    return arr * slope + intercept

def load_dicom_volume(folder_path, as_hu=False):
    if not os.path.isdir(folder_path): return None
    files = [f for f in os.listdir(folder_path) if f.lower().endswith('.dcm')]
    if not files: return None
    dss = [pydicom.dcmread(os.path.join(folder_path, name)) for name in files]
    dss.sort(key=lambda ds: float(getattr(ds, 'SliceLocation', getattr(ds, 'InstanceNumber', 0))))
    slices = [dicom_to_hu(ds) if as_hu else ds.pixel_array.astype(np.float32) for ds in dss]
    return np.stack(slices, axis=-1)

def find_roi_path(data_dir):
    for p in [os.path.join(data_dir, 'ILD_DB_volumeROIs'), data_dir]:
        if os.path.exists(p): return p
    return None

def _dcm_in(d):
    return os.path.isdir(d) and any(f.lower().endswith('.dcm') for f in os.listdir(d))

def _dir_has_ct(p):
    if _dcm_in(p): return True
    for name in ('volumeCT','ct','CT','VolumeCT'):
        if _dcm_in(os.path.join(p, name)): return True
    return False

def _dir_has_roi(p):
    for sub in ('roi_mask','ROI','roi'):
        if _dcm_in(os.path.join(p, sub)): return True
    return False

def _series_subdirs(p):
    out = []
    for s in sorted(os.listdir(p)):
        sp = os.path.join(p, s)
        if os.path.isdir(sp) and _dir_has_ct(sp) and _dir_has_roi(sp):
            out.append((s, sp))
    return out

def list_patient_units(roi_base):
    units = []
    if not os.path.isdir(roi_base): return units
    def add_patient(patient_dir, group, cohort):
        if _dir_has_ct(patient_dir) and _dir_has_roi(patient_dir):
            units.append({'load_path': patient_dir, 'uid': group, 'group': group,
                          'cohort': cohort, 'series': None, 'missing': False})
            return
        subs = _series_subdirs(patient_dir)
        if subs:
            for sname, spath in subs:
                units.append({'load_path': spath, 'uid': f'{group}__{sname}', 'group': group,
                              'cohort': cohort, 'series': sname, 'missing': False})
            return
        units.append({'load_path': patient_dir, 'uid': group, 'group': group,
                      'cohort': cohort, 'series': None, 'missing': True})
    for name in sorted(os.listdir(roi_base)):
        full = os.path.join(roi_base, name)
        if not os.path.isdir(full): continue
        if name == 'HRCT_pilot':
            for sub in sorted(os.listdir(full)):
                sub_full = os.path.join(full, sub)
                if os.path.isdir(sub_full): add_patient(sub_full, sub, 'pilot')
        else:
            add_patient(full, name, 'main')
    return units

def load_patient_ct(patient_path):
    for name in ['volumeCT','ct','CT','VolumeCT']:
        v = load_dicom_volume(os.path.join(patient_path, name), as_hu=True)
        if v is not None: return v
    return load_dicom_volume(patient_path, as_hu=True)

def load_roi_volume(patient_path):
    for sub in ['roi_mask','ROI','roi']:
        v = load_dicom_volume(os.path.join(patient_path, sub), as_hu=False)
        if v is not None: return v
    return None

def normalize_hu_volume(vol_hu, clip=HU_CLIP):
    lo, hi = clip
    v = np.clip(vol_hu.astype(np.float32), lo, hi)
    return ((v - lo) / (hi - lo + 1e-8)).astype(np.float32)

def resize_volume_inplane(vol, size, mode='bilinear'):
    h, w, d = vol.shape
    if h == size and w == size: return vol.astype(np.float32)
    out = np.zeros((size, size, d), dtype=np.float32)
    for z in range(d):
        t = torch.from_numpy(vol[:,:,z].astype(np.float32)).unsqueeze(0).unsqueeze(0)
        kwargs = {'size': (size, size), 'mode': mode}
        if mode in ('bilinear','bicubic'): kwargs['align_corners'] = False
        out[:,:,z] = F.interpolate(t, **kwargs).squeeze().numpy()
    return out

def roi_volume_to_seg_labels(roi_vol, lung_mask):
    labels = np.zeros(roi_vol.shape, dtype=np.uint8)
    lung = lung_mask > 0.5
    labels[lung] = 0
    r = np.rint(roi_vol).astype(np.int32)
    for cls_idx in range(1, 6):
        labels[lung & (r == cls_idx)] = cls_idx
    for raw_id, canonical in MEDGIFT_TO_CLS_FALLBACK.items():
        c = int(canonical)
        if c == 0: continue
        labels[lung & (r == int(raw_id))] = c
    return labels

def load_patient_volumes_3d(patient_roi_path, data_dir, inplane=128, pid_override=None):
    lung_base = LUNG_MASK_BASE
    pid = pid_override or os.path.basename(patient_roi_path.rstrip('/\\'))
    ct = load_patient_ct(patient_roi_path)
    roi = load_roi_volume(patient_roi_path)
    if ct is None or roi is None:
        return None
    if ct.shape != roi.shape:
        Z = min(ct.shape[-1], roi.shape[-1])
        ct = ct[:,:,:Z]; roi = roi[:,:,:Z]
    ct_norm = normalize_hu_volume(ct)
    if ct_norm.shape[-1] < CLS_PATCH_SIZE[0]:
        return None
    # Lung mask: prefer flat {pid}.npy from LUNG_MASK_BASE
    lung_mask_arr = None
    if lung_base:
        npy_path = os.path.join(lung_base, str(pid) + '.npy')
        if os.path.isfile(npy_path):
            lung_mask_arr = np.load(npy_path)
    if lung_mask_arr is None:
        lung_mask_arr = np.ones(ct_norm.shape, dtype=np.float32)
    # Resize all to inplane
    ct_rs = resize_volume_inplane(ct_norm, inplane)
    lung_rs = resize_volume_inplane(lung_mask_arr.astype(np.float32), inplane, mode='nearest')
    lung_binary = (lung_rs > 0.5).astype(np.uint8)
    roi_rs = resize_volume_inplane(roi.astype(np.float32), inplane, mode='nearest')
    seg_labels = roi_volume_to_seg_labels(roi_rs, lung_binary)
    # Transpose (H,W,D) -> (D,H,W) for patch mining
    ct_rs = np.transpose(ct_rs, (2, 0, 1))
    lung_binary = np.transpose(lung_binary, (2, 0, 1))
    seg_labels = np.transpose(seg_labels, (2, 0, 1))
    return {'ct_norm': ct_rs.astype(np.float32), 'lung_mask': lung_binary, 'seg_labels': seg_labels, 'pid': pid}
print('Data I/O functions defined.')

In [ ]:
# ── Patient enumeration (with class_counts like NB01) ──
roi_base = find_roi_path(MEDGIFT_ROOT)
print(f'ROI base: {roi_base}')
all_units = list_patient_units(roi_base) if roi_base else []
print(f'Total patient units found: {len(all_units)}')
missing = [u for u in all_units if u.get('missing')]
print(f'Missing CT/ROI: {len(missing)}')
valid_units = [u for u in all_units if not u.get('missing')]
print(f'Valid units: {len(valid_units)}')

patient_records = []
for u in valid_units:
    path = u['load_path']
    pid = u['uid']
    packed = load_patient_volumes_3d(path, MEDGIFT_ROOT, inplane=INPLANE, pid_override=pid)
    if packed is None:
        continue
    gt = packed['seg_labels']
    lung_b = packed['lung_mask'] > 0.5
    in_lung = gt[lung_b] if lung_b.any() else np.array([], dtype=np.int64)
    class_counts = np.bincount(in_lung.astype(np.int64), minlength=SEG_NUM_CLASSES)
    has_ild = int((class_counts[1:] > 0).any())
    patient_records.append({
        'pid': pid,
        'group': u['group'],
        'cohort': u.get('cohort', 'main'),
        'path': path,
        'has_ild': has_ild,
        'class_counts': class_counts.tolist(),
    })

print(f'Patient records built: {len(patient_records)}')
ild_count = sum(1 for r in patient_records if r['has_ild'])
print(f'ILD+ = {ild_count}, ILD- = {len(patient_records) - ild_count}')
print('class_counts attached for NB01-style stratified mining')

In [ ]:
# ── Patch mining (same dominant + 50/50 stratified protocol as NB01) ──
from collections import namedtuple
PatchRec = namedtuple('PatchRec', ['pid', 'origin', 'label', 'binary_label', 'hier_label', 'group'])

def extract_patch(vol, origin, patch_size):
    oz, oy, ox = origin
    pd, ph, pw = patch_size
    patch = vol[oz:min(vol.shape[0], oz + pd),
               oy:min(vol.shape[1], oy + ph),
               ox:min(vol.shape[2], ox + pw)]
    if patch.shape != (pd, ph, pw):
        out = np.zeros((pd, ph, pw), dtype=vol.dtype)
        out[:patch.shape[0], :patch.shape[1], :patch.shape[2]] = patch
        return out
    return patch

def patch_dominant_label(seg_patch, lung_patch):
    lung = lung_patch > 0.5
    lung_n = int(lung.sum())
    if lung_n == 0:
        return None
    counts = np.bincount(seg_patch[lung].astype(np.int64), minlength=SEG_NUM_CLASSES)
    path_counts = counts[1:].copy()
    if path_counts.sum() == 0:
        return 0
    return int(np.argmax(path_counts) + 1)

def patch_label_ok(seg_patch, lung_patch, target_cls):
    lung = lung_patch > 0.5
    lung_n = float(lung.sum())
    if lung_n <= 0:
        return False
    if (lung_n / float(seg_patch.size)) < MIN_PATCH_LUNG_FRAC:
        return False
    in_lung = seg_patch[lung]
    if target_cls == 0:
        return float((in_lung == 0).sum()) / lung_n >= MIN_NORMAL_LUNG_FRAC
    n_cls = float((in_lung == target_cls).sum())
    if n_cls < MIN_PATHOLOGY_VOXELS:
        return False
    if (n_cls / lung_n) < MIN_TARGET_CLASS_FRAC:
        return False
    if PATCH_LABEL_MODE == 'dominant':
        dom = patch_dominant_label(seg_patch, lung_patch)
        if dom is None or int(dom) != int(target_cls):
            return False
    return True

class VolumeCache:
    def __init__(self, data_dir, max_patients=None):
        self.data_dir = data_dir
        self.cache = {}
        self.max_patients = VOL_CACHE_PATIENTS if max_patients is None else max_patients

    def get(self, pid, path):
        if pid not in self.cache:
            if len(self.cache) >= self.max_patients:
                self.cache.pop(next(iter(self.cache)))
            packed = load_patient_volumes_3d(path, self.data_dir, inplane=INPLANE, pid_override=pid)
            if packed is not None:
                self.cache[pid] = packed
        return self.cache.get(pid)

def build_hierarchical_patch_bank(records, data_dir, n_patches=2400, seed=42, min_per_class=None):
    """NB01 protocol: dominant-label gates + 50/50 Normal/ILD + stratified 5-pathology ILD."""
    floor_per_cls = MIN_PATCHES_PER_CLASS if min_per_class is None else int(min_per_class)
    rng = np.random.RandomState(seed)
    vol_cache = VolumeCache(data_dir)
    bank = []
    seen_keys = set()

    ild_patients = [r for r in records if r['has_ild']]
    normal_patients = [r for r in records if not r['has_ild']]

    def collect_origins(seg_labels, lung_mask, cls_id):
        lung = lung_mask > 0.5
        if cls_id == 0:
            coords = np.argwhere((seg_labels == 0) & lung)
        else:
            coords = np.argwhere((seg_labels == cls_id) & lung)
        if len(coords) == 0:
            return []
        n_probe = min(len(coords), MAX_CLASS_ORIGIN_POOL * 4)
        if len(coords) > n_probe:
            coords = coords[rng.choice(len(coords), size=n_probe, replace=False)]
        origins = []
        pd, ph, pw = CLS_PATCH_SIZE
        d, h, w = seg_labels.shape
        for cz, cy, cx in coords:
            oz = int(np.clip(cz - pd // 2, 0, max(0, d - pd)))
            oy = int(np.clip(cy - ph // 2, 0, max(0, h - ph)))
            ox = int(np.clip(cx - pw // 2, 0, max(0, w - pw)))
            origin = (oz, oy, ox)
            if origin in seen_keys:
                continue
            lung_crop = extract_patch(lung_mask, origin, CLS_PATCH_SIZE)
            seg_crop = extract_patch(seg_labels, origin, CLS_PATCH_SIZE)
            if not patch_label_ok(seg_crop, lung_crop, cls_id):
                continue
            origins.append(origin)
            if len(origins) >= MAX_CLASS_ORIGIN_POOL:
                break
        return origins

    normal_budget = n_patches // 2
    ild_budget = n_patches - normal_budget

    collected = 0
    rng.shuffle(normal_patients)
    for rec in normal_patients:
        if collected >= normal_budget:
            break
        packed = vol_cache.get(rec['pid'], rec['path'])
        if packed is None:
            continue
        for origin in collect_origins(packed['seg_labels'], packed['lung_mask'], 0):
            if collected >= normal_budget:
                break
            key = (rec['pid'], origin, 0)
            if key in seen_keys:
                continue
            seen_keys.add(key)
            bank.append(PatchRec(rec['pid'], origin, 0, 0, 0, rec['group']))
            collected += 1

    pathology_collected = {c: 0 for c in range(1, 6)}
    ild_budget_per_class = max(floor_per_cls, ild_budget // 5)
    for _ in range(3):
        rng.shuffle(ild_patients)
        for rec in ild_patients:
            cc = rec.get('class_counts', [0] * SEG_NUM_CLASSES)
            for cls_id in range(1, 6):
                if cc[cls_id] == 0:
                    continue
                if pathology_collected[cls_id] >= ild_budget_per_class:
                    continue
                packed = vol_cache.get(rec['pid'], rec['path'])
                if packed is None:
                    continue
                for origin in collect_origins(packed['seg_labels'], packed['lung_mask'], cls_id):
                    if pathology_collected[cls_id] >= ild_budget_per_class:
                        break
                    key = (rec['pid'], origin, cls_id)
                    if key in seen_keys:
                        continue
                    seen_keys.add(key)
                    bank.append(PatchRec(
                        rec['pid'], origin, cls_id, 1, HIERARCHY_MAP[cls_id], rec['group']))
                    pathology_collected[cls_id] += 1

    binary_counts = np.bincount([r.binary_label for r in bank], minlength=2)
    orig_counts = np.bincount([r.label for r in bank], minlength=6)
    hier_counts = np.bincount([r.hier_label for r in bank], minlength=3)
    print(f'  Binary: Normal={binary_counts[0]} ILD={binary_counts[1]}')
    print(f'  6-class: {dict(zip(ORIGINAL_CLASS_NAMES, orig_counts.tolist()))}')
    print(f'  Hierarchical: {dict(zip(HIERARCHY_CLASSES, hier_counts.tolist()))}')
    return bank, vol_cache

print('NB01-style dominant stratified patch mining defined.')
print(f'  PATCH_LABEL_MODE={PATCH_LABEL_MODE} | 50/50 Normal/ILD | 5-class ILD quota')


In [ ]:
# ── Phase 2: Conservative Multi-Head Fine-Tuning ──

def train_phase2_v2(binary_ckpt_path, epochs_head=5, epochs_unfreeze=10, epochs_full=5):
    """Phase 2 with binary_head frozen throughout to preserve screening F1."""
    ck = torch.load(binary_ckpt_path, map_location=device, weights_only=False)
    model = HierarchicalEncoder3D(in_ch=1, use_se=True).to(device)
    model.load_state_dict(ck.get("model", ck), strict=False)
    print(f"Loaded Phase 1 checkpoint: {binary_ckpt_path}")

    for prm in model.binary_head.parameters():
        prm.requires_grad = False

    fold_name = os.path.splitext(os.path.basename(binary_ckpt_path))[0]

    seed = GLOBAL_SEED
    rng = np.random.RandomState(seed)
    groups = list(set(r["group"] for r in patient_records))
    rng.shuffle(groups)
    group_has_ild = [1 if any(r["has_ild"] for r in patient_records if r["group"] == g) else 0 for g in groups]
    n_splits = min(N_FOLDS, len(groups))
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    fold_idx = 0
    m = re.search(r"R(\d+)F(\d+)", fold_name)
    if m:
        fold_idx = int(m.group(2)) % n_splits
    splits = list(skf.split(groups, group_has_ild))
    train_idx, val_idx = splits[fold_idx]
    train_groups = [groups[i] for i in train_idx]
    val_groups = [groups[i] for i in val_idx]
    train_recs = [r for r in patient_records if r["group"] in train_groups]
    val_recs = [r for r in patient_records if r["group"] in val_groups]
    print(f"Phase 2 split fold={fold_idx}: train={len(train_recs)} val={len(val_recs)}")

    train_bank, train_vc = build_hierarchical_patch_bank(
        train_recs, MEDGIFT_ROOT, n_patches=PATCHES_PER_EPOCH, seed=seed + 17)
    val_bank, val_vc = build_hierarchical_patch_bank(
        val_recs, MEDGIFT_ROOT, n_patches=VAL_PATCHES, seed=seed + 117,
        min_per_class=max(10, VAL_PATCHES // 6))

    class PatchDatasetP2(Dataset):
        def __init__(self, bank, vol_cache, pid_to_rec, augment=True):
            self.bank = bank
            self.vol_cache = vol_cache
            self.pid_to_rec = pid_to_rec
            self.augment = augment
        def __len__(self):
            return len(self.bank)
        def __getitem__(self, idx):
            rec = self.bank[idx]
            packed = self.vol_cache.get(rec.pid, self.pid_to_rec[rec.pid]["path"])
            if packed is None:
                return torch.zeros(1, *CLS_PATCH_SIZE), torch.tensor(0), torch.tensor(0), torch.tensor(0)
            ct = packed["ct_norm"]
            x = extract_patch(ct, rec.origin, CLS_PATCH_SIZE)
            x = torch.from_numpy(x).unsqueeze(0).float()
            if self.augment:
                if torch.rand(1).item() > 0.5: x = x.flip(-1)
                if torch.rand(1).item() > 0.5: x = x.flip(-2)
                if torch.rand(1).item() > 0.5:
                    x = torch.rot90(x, int(torch.randint(0, 4, (1,)).item()), dims=[-2, -1])
            return x, torch.tensor(rec.binary_label), torch.tensor(rec.hier_label), torch.tensor(rec.label)

    pid_tr = {r["pid"]: r for r in train_recs}
    pid_va = {r["pid"]: r for r in val_recs}
    train_loader = DataLoader(
        PatchDatasetP2(train_bank, train_vc, pid_tr, augment=True),
        batch_size=FEAT_BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
    val_loader = DataLoader(
        PatchDatasetP2(val_bank, val_vc, pid_va, augment=False),
        batch_size=FEAT_BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

    # --- Losses ---
    bin_counts = np.bincount([r.binary_label for r in train_bank], minlength=2).astype(float)
    bin_freq = bin_counts / max(bin_counts.sum(), 1)
    bin_alpha = torch.tensor(1.0 / np.maximum(bin_freq, 1e-6), dtype=torch.float32).to(device)
    bin_alpha = bin_alpha / bin_alpha.sum() * 2
    path_mask_arr = np.array([r.label for r in train_bank]) > 0
    if path_mask_arr.sum() > 0:
        path_counts = np.bincount([r.label - 1 for r in train_bank if r.label > 0], minlength=5).astype(float)
        path_freq = path_counts / max(path_counts.sum(), 1)
        path_alpha = torch.tensor(1.0 / np.maximum(path_freq, 1e-6), dtype=torch.float32).to(device)
        path_alpha = path_alpha / path_alpha.sum() * 5
    else:
        path_alpha = None
    bin_criterion = WeightedFocalLoss(alpha=bin_alpha, gamma=2.0)
    hier_criterion = LabelSmoothCrossEntropy(smoothing=0.1)
    path_criterion = WeightedFocalLoss(alpha=path_alpha, gamma=2.0) if path_alpha is not None else nn.CrossEntropyLoss()

    n_total = epochs_head + epochs_unfreeze + epochs_full
    best_state = None
    best_score = -1.0
    stale = 0
    phase2_path = None

    # Phase 2a: hier+path heads only, binary frozen
    for prm in model.parameters(): prm.requires_grad = False
    for prm in model.binary_head.parameters(): prm.requires_grad = False
    for prm in model.hier_head.parameters(): prm.requires_grad = True
    for prm in model.path_head.parameters(): prm.requires_grad = True
    opt = torch.optim.AdamW(
        list(model.hier_head.parameters()) + list(model.path_head.parameters()),
        lr=1e-4, weight_decay=FINETUNE_WD)
    print(f"Phase 2a: hier+path heads only (binary frozen), {epochs_head} epochs, lr=1e-4")

    for epoch in range(n_total):
        if epoch == epochs_head:
            # Phase 2b: unfreeze layer4, binary still frozen
            set_trainable_blocks(model, ("layer4",))
            for prm in model.binary_head.parameters(): prm.requires_grad = False
            for prm in model.hier_head.parameters(): prm.requires_grad = True
            for prm in model.path_head.parameters(): prm.requires_grad = True
            enc_params = [p for n, p in model.named_parameters() if p.requires_grad
                          and not n.startswith(("binary_head", "hier_head", "path_head"))]
            opt = torch.optim.AdamW([
                {"params": enc_params, "lr": 3e-5},
                {"params": model.hier_head.parameters(), "lr": 1e-4},
                {"params": model.path_head.parameters(), "lr": 1e-4},
            ], weight_decay=FINETUNE_WD)
            print(f"Phase 2b: unfreeze layer4, {epochs_unfreeze} epochs")
            stale = 0

        if epoch == epochs_head + epochs_unfreeze:
            # Phase 2c: carefully unfreeze binary head
            for prm in model.binary_head.parameters(): prm.requires_grad = True
            opt = torch.optim.AdamW([
                {"params": enc_params, "lr": 1e-5},
                {"params": model.binary_head.parameters(), "lr": 1e-5},
                {"params": model.hier_head.parameters(), "lr": 1e-5},
                {"params": model.path_head.parameters(), "lr": 1e-5},
            ], weight_decay=FINETUNE_WD)
            print(f"Phase 2c: all heads unfrozen, {epochs_full} epochs, lr=1e-5")
            stale = 0

        # Training step
        model.train()
        for step, (x, y_bin, y_hier, y_orig) in enumerate(train_loader):
            x = x.to(device); y_bin = y_bin.to(device); y_hier = y_hier.to(device); y_orig = y_orig.to(device)
            feat = model.extract_features(x)
            with torch.no_grad():
                bin_criterion(model.binary_head(feat), y_bin)
            hier_loss = 0.5 * hier_criterion(model.hier_head(feat), y_hier)
            path_mask = y_orig > 0
            if path_mask.sum() > 0:
                path_loss = 0.3 * path_criterion(model.path_head(feat[path_mask]), y_orig[path_mask] - 1)
            else:
                path_loss = torch.tensor(0.0, device=device)
            loss = (hier_loss + path_loss) / GRAD_ACCUM_STEPS
            loss.backward()
            if ((step + 1) % GRAD_ACCUM_STEPS == 0) or ((step + 1) == len(train_loader)):
                opt.step(); opt.zero_grad(set_to_none=True)

        # Validation
        model.eval()
        yb_t, yb_p, yh_t, yh_p, yp_t, yp_p = [], [], [], [], [], []
        with torch.no_grad():
            for x, y_bin, y_hier, y_orig in val_loader:
                x = x.to(device)
                feat = model.extract_features(x)
                pb = F.softmax(model.binary_head(feat), dim=1)
                ph = F.softmax(model.hier_head(feat), dim=1)
                pp = F.softmax(model.path_head(feat), dim=1)
                yb_t.extend(y_bin.numpy().tolist())
                yb_p.extend(pb.argmax(1).cpu().numpy().tolist())
                yh_t.extend(y_hier.numpy().tolist())
                yh_p.extend(ph.argmax(1).cpu().numpy().tolist())
                yp_t.extend(y_orig.numpy().tolist())
                yp_p.extend(pp.argmax(1).cpu().numpy().tolist())

        bin_f1 = float(f1_score(yb_t, yb_p, average="binary", zero_division=0))
        hier_f1 = float(f1_score(yh_t, yh_p, average="macro", zero_division=0))
        path_f1_val = 0.0
        pmask = [i for i, v in enumerate(yp_t) if v > 0]
        if pmask:
            path_f1_val = float(f1_score(
                [yp_t[i] for i in pmask], [yp_p[i] for i in pmask],
                labels=list(range(1, 6)), average="macro", zero_division=0))

        combined = bin_f1 + 0.3 * hier_f1 + 0.1 * path_f1_val

        if epoch >= epochs_head + epochs_unfreeze and bin_f1 < 0.85:
            print(f"  Phase 2c: binary F1={bin_f1:.4f} < 0.85, reverting to best state")
            if best_state:
                model.load_state_dict(best_state, strict=False)
            break

        if combined > best_score:
            best_score = combined
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            stale = 0
        else:
            stale += 1
            if epoch >= epochs_head and stale >= FINETUNE_PATIENCE:
                print(f"  Early stop at epoch {epoch+1}")
                break

        if (epoch + 1) % 3 == 0 or epoch == 0:
            print(f"  ep {epoch+1}/{n_total} binF1={bin_f1:.4f} hierF1={hier_f1:.4f} pathF1={path_f1_val:.4f}")

    # Save
    if best_state:
        model.load_state_dict(best_state, strict=False)
        phase2_path = os.path.join(MODELS_DIR, f"phase2_{fold_name}.pth")
        model.eval()
        yb_t, yb_p = [], []
        with torch.no_grad():
            for x, y_bin, _, _ in val_loader:
                x = x.to(device)
                pb = F.softmax(model.binary_head(model.extract_features(x)), dim=1)
                yb_t.extend(y_bin.numpy().tolist())
                yb_p.extend(pb.argmax(1).cpu().numpy().tolist())
        final_bin_f1 = float(f1_score(yb_t, yb_p, average="binary", zero_division=0))
        torch.save({"model": best_state, "best_bin_f1": final_bin_f1, "best_combined": best_score}, phase2_path)
        print(f"\nPhase 2 saved: {phase2_path}")
        print(f"  Final patch binary F1={final_bin_f1:.4f}  (target: >= 0.85)")
    return model, phase2_path


# --- Resolve best Phase 1 checkpoint ---
def _resolve_hier_checkpoint():
    best_path, best_f1 = None, -1.0
    names = sorted(os.listdir(MODELS_DIR)) if os.path.isdir(MODELS_DIR) else []
    for name in names:
        if not (name.startswith('hierarchical_fold_') and name.endswith('.pth')):
            continue
        path = os.path.join(MODELS_DIR, name)
        try:
            ck = torch.load(path, map_location='cpu', weights_only=False)
            f1 = float(ck.get('best_bin_f1', -1))
            if f1 > best_f1:
                best_f1, best_path = f1, path
        except Exception:
            if best_path is None: best_path = path
    return best_path, best_f1

phase2_ckpt_path, p2_f1 = _resolve_hier_checkpoint()
print(f"Best Phase 1 checkpoint: {phase2_ckpt_path} (F1={p2_f1})")

In [ ]:
# ── Full-volume cascade inference (for Phase 2 model) ──

def cuda_cleanup():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def cascade_classify_volume(ct_norm, lung_mask, model):
    """Sliding-window Softmax voting -> 3D label map + patient decision."""
    model.eval()
    d, h, w = ct_norm.shape
    pd, ph, pw = CLS_PATCH_SIZE
    stride = INFER_DENSE_STRIDE
    vote = np.zeros((SEG_NUM_CLASSES, d, h, w), dtype=np.float32)
    weight = np.zeros((d, h, w), dtype=np.float32)
    coords = np.argwhere(lung_mask > 0.5)
    if len(coords) == 0:
        return np.zeros((d, h, w), dtype=np.uint8), {'hist': [0]*SEG_NUM_CLASSES}
    z0, y0, x0 = coords.min(axis=0)
    z1, y1, x1 = coords.max(axis=0) + 1

    def _get_binary_proba(patch_tensor):
        with torch.no_grad():
            f = model.extract_features(patch_tensor)
            pb = F.softmax(model.binary_head(f), dim=1)
            pp = F.softmax(model.path_head(f), dim=1)
        proba = np.zeros(SEG_NUM_CLASSES, dtype=np.float32)
        p_ild = float(pb[0, 1].item())
        proba[0] = 1.0 - p_ild
        proba[1:6] = pp[0].cpu().numpy() * p_ild
        return proba

    zs = list(range(max(0, z0 - pd // 2), max(1, min(d, z1) - pd + 1), stride[0])) or [0]
    ys = list(range(max(0, y0 - ph // 2), max(1, min(h, y1) - ph + 1), stride[1])) or [0]
    xs = list(range(max(0, x0 - pw // 2), max(1, min(w, x1) - pw + 1), stride[2])) or [0]

    processed = 0
    for oz in tqdm(zs, desc='Cascade', leave=False):
        for oy in ys:
            for ox in xs:
                if processed >= INFER_MAX_PATCHES: break
                lc = extract_patch(lung_mask, (oz, oy, ox), CLS_PATCH_SIZE)
                if lc.mean() < 0.2: continue
                patch = extract_patch(ct_norm, (oz, oy, ox), CLS_PATCH_SIZE)
                t = torch.from_numpy(patch).unsqueeze(0).unsqueeze(0).float().to(device)
                proba = _get_binary_proba(t)
                inside = lc > 0.5
                dd, hh, ww = lc.shape
                for c in range(SEG_NUM_CLASSES):
                    vote[c, oz:oz+dd, oy:oy+hh, ox:ox+ww][inside] += float(proba[c])
                weight[oz:oz+dd, oy:oy+hh, ox:ox+ww][inside] += 1.0
                processed += 1

    vol_map = np.zeros((d, h, w), dtype=np.uint8)
    inside = lung_mask > 0.5
    if inside.any():
        probs = vote[:, inside] / np.maximum(weight[inside], 1e-6)
        vol_map[inside] = np.argmax(probs, axis=0).astype(np.uint8)
    # Median filter
    if vol_map.ndim == 3:
        from scipy import ndimage
        vol_map_smooth = ndimage.median_filter(vol_map.astype(np.float32), size=3)
        vol_map = np.rint(vol_map_smooth).astype(np.uint8)

    # Patient-level decision (dual threshold)
    lung_vox = int(inside.sum())
    hist = np.bincount(vol_map[inside].astype(np.int64), minlength=SEG_NUM_CLASSES) if inside.any() else np.zeros(SEG_NUM_CLASSES, dtype=np.int64)
    pathology_vox = int(hist[1:].sum())
    path_frac = pathology_vox / max(lung_vox, 1)

    # Mean ILD Softmax probability
    n_ild_patches = max(processed, 1)
    mean_ild_prob = float((vote[1:].sum(axis=0)[inside]).sum() / max(weight[inside].sum(), 1)) if inside.any() else 0.0

    pred_binary = 1 if (path_frac >= CASCADE_PATH_THRESH or mean_ild_prob >= CASCADE_PROB_THRESH) else 0
    pred_path = int(np.argmax(hist[1:])) + 1 if hist[1:].sum() > 0 else 0
    hier_counts = np.zeros(N_HIER_CLASSES, dtype=np.int64)
    for oc in range(SEG_NUM_CLASSES):
        hier_counts[HIERARCHY_MAP[oc]] += hist[oc]
    pred_hier = int(np.argmax(hier_counts[1:]) + 1) if hier_counts[1:].sum() > 0 else 0

    fibrotic_labels = {2, 5}
    fibrotic_vox = sum(hist[i] for i in fibrotic_labels if i < len(hist))
    nonfibrotic_vox = sum(hist[i] for i in range(1, SEG_NUM_CLASSES) if i not in fibrotic_labels and i < len(hist))

    meta = {
        'pred_binary': pred_binary, 'pred_hier': pred_hier, 'pred_path': pred_path,
        'n_patches': processed, 'lung_voxels': lung_vox,
        'pathology_voxels': pathology_vox, 'pathology_frac': float(path_frac),
        'fibrotic_frac': float(fibrotic_vox / max(lung_vox, 1)),
        'nonfibrotic_frac': float(nonfibrotic_vox / max(lung_vox, 1)),
        'mean_ild_prob': float(mean_ild_prob),
        'hist': [int(hist[i]) for i in range(SEG_NUM_CLASSES)],
    }
    return vol_map, meta


def run_full_volume_cascade(records, ckpt_path=None, max_patients=0):
    if ckpt_path is None:
        ckpt_path, best_f1 = _resolve_hier_checkpoint()
        print(f'Cascade checkpoint: {ckpt_path} (best_bin_f1={best_f1})')
    else:
        print(f'Cascade checkpoint: {ckpt_path}')
    if not ckpt_path or not os.path.isfile(ckpt_path):
        print('SKIP cascade: no checkpoint found.')
        return None

    model = HierarchicalEncoder3D(in_ch=1, use_se=True).to(device)
    ck = torch.load(ckpt_path, map_location=device, weights_only=False)
    state = ck.get('model', ck)
    model.load_state_dict(state, strict=False)
    model.eval()

    recs = list(records)
    if max_patients and max_patients > 0:
        recs = recs[:max_patients]

    rows = []
    vc = VolumeCache(MEDGIFT_ROOT, max_patients=2)
    for rec in tqdm(recs, desc='Full-volume cascade'):
        packed = vc.get(rec['pid'], rec['path'])
        if packed is None: continue
        ct = packed['ct_norm']
        lung = packed['lung_mask']
        gt_seg = packed.get('seg_labels')
        vol_map, meta = cascade_classify_volume(ct, lung, model)

        gt_binary = int(rec.get('has_ild', 0))
        lung_b = lung > 0.5
        if gt_seg is not None and lung_b.any():
            counts = np.bincount(gt_seg[lung_b].astype(np.int64), minlength=SEG_NUM_CLASSES)
            gt_path = int(np.argmax(counts[1:]) + 1) if counts[1:].sum() > 0 else 0
            hier_counts = np.zeros(N_HIER_CLASSES, dtype=np.int64)
            for oc in range(SEG_NUM_CLASSES):
                hier_counts[HIERARCHY_MAP[oc]] += counts[oc]
            gt_hier = int(np.argmax(hier_counts[1:]) + 1) if hier_counts[1:].sum() > 0 else 0
        else:
            gt_path, gt_hier = -1, -1

        if CASCADE_SAVE_MAPS:
            np.savez_compressed(
                os.path.join(CASCADE_DIR, f"map_{rec['pid']}.npz"),
                pathology_map=vol_map.astype(np.int16),
                lung_mask=(lung > 0.5).astype(np.uint8),
                hist=np.array(meta['hist'], dtype=np.int64))

        rows.append({
            'pid': rec['pid'], 'group': rec['group'], 'cohort': rec.get('cohort'),
            'gt_binary': gt_binary, 'pred_binary': meta['pred_binary'],
            'gt_path': gt_path, 'gt_hier': gt_hier,
            'pred_path': meta['pred_path'], 'pred_hier': meta['pred_hier'],
            'n_patches': meta['n_patches'], 'lung_voxels': meta['lung_voxels'],
            'pathology_voxels': meta['pathology_voxels'], 'pathology_frac': meta['pathology_frac'],
            'fibrotic_frac': meta['fibrotic_frac'], 'nonfibrotic_frac': meta['nonfibrotic_frac'],
            'mean_ild_prob': meta['mean_ild_prob'],
            **{f'vox_{ORIGINAL_CLASS_NAMES[i].replace(" ", "_")}': meta['hist'][i] for i in range(SEG_NUM_CLASSES)},
        })
        del packed, vol_map
        cuda_cleanup()

    del model; cuda_cleanup()

    df = pd.DataFrame(rows)
    out_csv = os.path.join(EXPORTS_DIR, 'hierarchical_cascade_patients.csv')
    df.to_csv(out_csv, index=False)

    summary = {
        'protocol': 'CT_lungmask_sliding_window_patch_softmax_3D_map_then_classify',
        'checkpoint': ckpt_path,
        'stride': list(INFER_DENSE_STRIDE),
        'patch_size': list(CLS_PATCH_SIZE),
        'n_patients': int(len(df)),
        'decision_rule': f'path_frac>={CASCADE_PATH_THRESH} OR mean_ild_prob>={CASCADE_PROB_THRESH}',
    }
    if len(df) and 'gt_binary' in df.columns:
        yb, pb = df['gt_binary'].astype(int), df['pred_binary'].astype(int)
        summary['patient_binary_f1'] = float(f1_score(yb, pb, zero_division=0))
        summary['patient_binary_acc'] = float(accuracy_score(yb, pb))
        if 'gt_hier' in df.columns and 'pred_hier' in df.columns:
            yh, ph = df['gt_hier'].astype(int), df['pred_hier'].astype(int)
            summary['patient_hier_macro_f1'] = float(f1_score(yh, ph, average='macro', zero_division=0))
        if 'gt_path' in df.columns and 'pred_path' in df.columns:
            pm = df['gt_path'] > 0
            if pm.any():
                summary['patient_path_macro_f1_all'] = float(f1_score(
                    df.loc[pm, 'gt_path'], df.loc[pm, 'pred_path'],
                    labels=list(range(1, 6)), average='macro', zero_division=0))

    out_json = os.path.join(EXPORTS_DIR, 'hierarchical_cascade_summary.json')
    with open(out_json, 'w', encoding='utf-8') as f:
        json.dump(summary, f, indent=2)

    print(f'Patients={len(df)} | saved {out_csv}')
    print(f'Summary: {out_json}')
    if 'patient_binary_f1' in summary:
        print(f"Patient binary F1={summary['patient_binary_f1']:.4f} Acc={summary['patient_binary_acc']:.4f}")
    if 'patient_hier_macro_f1' in summary:
        print(f"Patient hier Macro-F1={summary['patient_hier_macro_f1']:.4f}")
    if 'patient_path_macro_f1_all' in summary:
        print(f"Patient path Macro-F1={summary['patient_path_macro_f1_all']:.4f}")
    return {'summary': summary, 'df': df}

print('Cascade functions defined.')

In [ ]:
# ── Execute Phase 2 ──
print("=" * 60)
print("  PHASE 2: Conservative Multi-Head Fine-Tuning")
print("=" * 60)

phase2_ckpt_path, p2_f1 = _resolve_hier_checkpoint()
print(f"\nBest Phase 1 checkpoint: {phase2_ckpt_path} (F1={p2_f1})")

if not phase2_ckpt_path or not os.path.isfile(phase2_ckpt_path):
    print("ERROR: No Phase 1 checkpoint found! Train Phase 1 first.")
else:
    model_p2, saved_path = train_phase2_v2(phase2_ckpt_path)

    if saved_path and os.path.isfile(saved_path):
        print("\n" + "=" * 60)
        print("  CASCADE: Full-volume inference with Phase 2 model")
        print("=" * 60)
        cascade_p2 = run_full_volume_cascade(
            patient_records, ckpt_path=saved_path, max_patients=CASCADE_MAX_PATIENTS)
        if cascade_p2:
            s = cascade_p2["summary"]
            print("\n" + "=" * 60)
            print("  PHASE 2 RESULTS")
            print("=" * 60)
            bf1 = s.get("patient_binary_f1", "N/A")
            bacc = s.get("patient_binary_acc", "N/A")
            hf1 = s.get("patient_hier_macro_f1", "N/A")
            pf1 = s.get("patient_path_macro_f1_all", "N/A")
            print(f"  Binary F1       = {bf1}")
            print(f"  Binary Acc      = {bacc}")
            print(f"  3-class Hier F1 = {hf1}")
            print(f"  5-class Path F1 = {pf1}")
            if isinstance(bf1, (int, float)) and bf1 >= 0.85:
                print("\n  STATUS: OK. Binary preserved. Update results.tex.")
            else:
                print("\n  STATUS: Binary dropped. Use Phase 1 for paper.")
    else:
        print("Phase 2 training did not produce a checkpoint.")

print("\nDone.")